In [2]:
import polars as pl

df = pl.read_csv("../data/vehicle_data_2025_07_22.csv")

In [2]:
# For numeric typ rows, check fzg_nr — the vehicle number might reveal the type
# DVB trams: 2501-2593 (NGT6DD), 2601-2640 (NGTD8DD), 2701-2723 (NGT8DD)
#            2801-2843 (NGTD12DD), 2901-2989 (NGTDX)
# DVB buses: typically 4xx or 5xx numbers
print("Numeric typ: fzg_nr range per typ_text")
print(
    df.filter(pl.col("typ").str.contains(r"^\d+$"))
    .with_columns(pl.col("fzg_nr").str.replace_all(" ", "").alias("fzg_nr_clean"))
    .group_by(["typ", "typ_text"])
    .agg([
        pl.col("fzg_nr_clean").min().alias("fzg_nr_min"),
        pl.col("fzg_nr_clean").max().alias("fzg_nr_max"),
        pl.col("fzg_id").n_unique().alias("num_vehicles"),
    ])
    .sort("num_vehicles", descending=True)
)

Numeric typ: fzg_nr range per typ_text
shape: (16, 5)
┌─────┬──────────┬────────────┬────────────┬──────────────┐
│ typ ┆ typ_text ┆ fzg_nr_min ┆ fzg_nr_max ┆ num_vehicles │
│ --- ┆ ---      ┆ ---        ┆ ---        ┆ ---          │
│ str ┆ str      ┆ str        ┆ str        ┆ u32          │
╞═════╪══════════╪════════════╪════════════╪══════════════╡
│ 8   ┆ ENT      ┆ null       ┆ null       ┆ 388          │
│ 67  ┆ TETR     ┆ null       ┆ null       ┆ 291          │
│ 9   ┆ ANZ      ┆ null       ┆ null       ┆ 272          │
│ 14  ┆ BES      ┆ null       ┆ null       ┆ 203          │
│ 3   ┆ DV       ┆ null       ┆ null       ┆ 185          │
│ …   ┆ …        ┆ …          ┆ …          ┆ …            │
│ 111 ┆ iSD      ┆ null       ┆ null       ┆ 27           │
│ 119 ┆ iFGZ     ┆ null       ┆ null       ┆ 26           │
│ 3   ┆ IRIS     ┆ null       ┆ null       ┆ 13           │
│ 20  ┆ GPS      ┆ null       ┆ null       ┆ 1            │
│ 6   ┆ ZBM      ┆ null       ┆ null       ┆ 1

In [3]:
# Cross-check: for the named tram types, what fzg_nr range do they have?
# This helps us understand what fzg_nr range corresponds to which tram type
print("Named tram types: fzg_nr range")
print(
    df.filter(pl.col("typ").str.contains("NGT|NGTDX|T4D"))
    .with_columns(pl.col("fzg_nr").str.replace_all(" ", "").alias("fzg_nr_clean"))
    .group_by(["typ", "typ_nr"])
    .agg([
        pl.col("fzg_nr_clean").min().alias("fzg_nr_min"),
        pl.col("fzg_nr_clean").max().alias("fzg_nr_max"),
        pl.col("fzg_id").n_unique().alias("num_vehicles"),
    ])
    .sort("typ")
)

Named tram types: fzg_nr range
shape: (9, 5)
┌────────────┬────────┬────────────┬────────────┬──────────────┐
│ typ        ┆ typ_nr ┆ fzg_nr_min ┆ fzg_nr_max ┆ num_vehicles │
│ ---        ┆ ---    ┆ ---        ┆ ---        ┆ ---          │
│ str        ┆ f64    ┆ str        ┆ str        ┆ u32          │
╞════════════╪════════╪════════════╪════════════╪══════════════╡
│ NGT 6 DD   ┆ 9.0    ┆ 232500-5   ┆ 232544-8   ┆ 38           │
│ NGT 6 DD 2 ┆ 10.0   ┆ 232580-0   ┆ 232593-8   ┆ 14           │
│ NGT 8 DD   ┆ 13.0   ┆ 232700-7   ┆ 232723-2   ┆ 24           │
│ NGTD 12 DD ┆ 14.0   ┆ 232800-8   ┆ 232843-4   ┆ 44           │
│ NGTD 8 DD  ┆ 15.0   ┆ 232600-6   ┆ 232640-8   ┆ 41           │
│ NGTDX      ┆ 26.0   ┆ 232900-9   ┆ 232931-4   ┆ 32           │
│ NGTDX ZR   ┆ 27.0   ┆ 232980-4   ┆ 232989-4   ┆ 10           │
│ T4D MT     ┆ 3.0    ┆ 224263-8   ┆ 224277-5   ┆ 3            │
│ T4D MT     ┆ 1.0    ┆ 201317-7   ┆ 201318-5   ┆ 2            │
└────────────┴────────┴────────────┴─────────

In [4]:
# Check typ_text meanings:
# ENT=Einsatznotiz? ANZ=Anzeige? BES=Besetzt? 
# These might be STATUS codes, not vehicle types
# Let's check if a single fzg_id appears with BOTH numeric typ AND named typ
named_typ_vehicles = (
    df.filter(pl.col("typ").str.contains("NGT|NGTDX|T4D|Urbino|MAN"))
    ["fzg_id"].unique()
)

print("Do vehicles with named typ also appear with numeric typ?")
overlap = df.filter(
    pl.col("fzg_id").is_in(named_typ_vehicles) &
    pl.col("typ").str.contains(r"^\d+$")
)
print(f"Rows where named-typ vehicle has numeric typ: {len(overlap):,}")
if len(overlap) > 0:
    print(overlap.select(["fzg_id", "typ", "typ_text", "fzg_nr"]).head(10))

Do vehicles with named typ also appear with numeric typ?
Rows where named-typ vehicle has numeric typ: 85,741
shape: (10, 4)
┌────────┬─────┬──────────┬────────┐
│ fzg_id ┆ typ ┆ typ_text ┆ fzg_nr │
│ ---    ┆ --- ┆ ---      ┆ ---    │
│ f64    ┆ str ┆ str      ┆ str    │
╞════════╪═════╪══════════╪════════╡
│ 1843.0 ┆ 8   ┆ ENT      ┆ null   │
│ 1575.0 ┆ 14  ┆ BES      ┆ null   │
│ 1575.0 ┆ 14  ┆ BES      ┆ null   │
│ 776.0  ┆ 8   ┆ ENT      ┆ null   │
│ 2193.0 ┆ 8   ┆ ENT      ┆ null   │
│ 1808.0 ┆ 8   ┆ ENT      ┆ null   │
│ 2375.0 ┆ 8   ┆ ENT      ┆ null   │
│ 1606.0 ┆ 8   ┆ ENT      ┆ null   │
│ 1875.0 ┆ 8   ┆ ENT      ┆ null   │
│ 1800.0 ┆ 8   ┆ ENT      ┆ null   │
└────────┴─────┴──────────┴────────┘


/var/folders/jg/lxcfp6vx34d70nm8knsjxpsc0000gn/T/ipykernel_50996/1707298821.py:11: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  overlap = df.filter(


In [5]:
df.filter(pl.col("typ") == "NGT 6 DD 2")["fzg_nr"].unique().sort()

fzg_nr
str
"""232 580-0"""
"""232 581-7"""
"""232 582-5"""
"""232 583-3"""
"""232 584-1"""
…
"""232 589-0"""
"""232 590-5"""
"""232 591-3"""


In [11]:

df.filter(pl.col("typ") == "MBeCitaroG")["fzg_nr"].unique().sort()

fzg_nr
str
"""464 001-3"""
"""464 002-1"""
"""464 003-8"""
"""464 004-6"""
"""464 005-4"""
…
"""464 014-2"""
"""464 015-0"""
"""464 016-7"""


In [10]:
# All non-numeric typ values — these are real vehicle types
real_vehicles = df.filter(
    ~pl.col("typ").str.contains(r"^\d+$") & pl.col("typ").is_not_null()
)

result = (
    real_vehicles
    .group_by(["typ", "typ_nr"])
    .agg([
        pl.col("fzg_id").n_unique().alias("num_vehicles"),
        pl.col("kapazitaet").first(),
        pl.col("laenge").first(),
        pl.col("masse").first(),
    ])
    .sort("num_vehicles", descending=True)
)
result.write_csv("vehicle_types_all.csv")
print("saved")

saved
